# R7 — Weekly Rejection Rate + Alert

The operations team wants to be told when a plan's weekly rejection rate gets uncomfortably high — not have to remember to check.

* **Weekly** = the Monday-starting week of the **submission** date, **decided claims only** (approved + rejected + settled; a claim still at `submitted` belongs on neither side of the fraction).
* **Rejection rate** = rejected ÷ decided.
* The dataset is frozen as of **2026-06-30**, so the alert asks about the **fixed week starting 2026-06-22** (Monday), never "the latest week" — otherwise it would quietly stop meaning anything once time moves past the frozen window.

In [0]:
%sql
-- weekly_rejection_rate: Monday-starting week of the SUBMISSION date, decided claims only
CREATE OR REPLACE TABLE claims.analytics.weekly_rejection_rate AS
SELECT
  p.plan_code,
  d.week_start,
  COUNT(*)                                            AS decided_count,
  SUM(CASE WHEN f.status = 'rejected' THEN 1 ELSE 0 END) AS rejected_count,
  ROUND(SUM(CASE WHEN f.status = 'rejected' THEN 1 ELSE 0 END) / COUNT(*), 4) AS rejection_rate
FROM claims.analytics.fact_claims f
JOIN claims.analytics.dim_plan p ON f.plan_sk = p.plan_sk
JOIN claims.analytics.dim_date d ON f.submitted_date_key = d.date_key
WHERE f.status IN ('approved', 'rejected', 'settled')
GROUP BY p.plan_code, d.week_start
ORDER BY p.plan_code, d.week_start;

In [0]:
%sql
-- The alert's query: ONE row — the highest plan rejection rate in the FIXED week 2026-06-22.
-- (Alert claims_weekly_rejection_alert fires when this value is > 0.18.)
SELECT
  MAX(rejection_rate) AS highest_plan_rejection_rate
FROM claims.analytics.weekly_rejection_rate
WHERE week_start = DATE'2026-06-22';

In [0]:
%sql
-- Sanity view of the fixed week: which plans are near or above the 0.18 threshold
SELECT plan_code, decided_count, rejected_count, rejection_rate
FROM claims.analytics.weekly_rejection_rate
WHERE week_start = DATE'2026-06-22'
ORDER BY rejection_rate DESC;